# Relabeling and Dataset Update Pipeline

This notebook downloads the original generated Chain of Thoughts, runs the updated `label_hops.py` (which fixes the blank entities bug), regenerates the counterfactuals and splits, and uploads the completely clean dataset back to Hugging Face.

In [ ]:
!pip install sentence-transformers datasets huggingface_hub SPARQLWrapper nnsight -q

### 1. Clone GitHub Repository & Set HF Token

In [ ]:
import os, sys, subprocess
from huggingface_hub import login

# Put your HF Token here (or use Kaggle Secrets)
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
except:
    HF_TOKEN = "your_hf_token_here"
    
login(token=HF_TOKEN)
os.environ['HF_TOKEN'] = HF_TOKEN

REPO_URL = 'https://github.com/ANLPproject/LinguaFranca.git'
WORK_DIR = '/kaggle/working/LinguaFranca'
if not os.path.exists(WORK_DIR):
    subprocess.run(['git', 'clone', '-b', 'main', REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(['git', '-C', WORK_DIR, 'pull'], check=True)
    
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print('Working directory:', os.getcwd())

### 2. Download generated_cot.jsonl from Hugging Face

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

# Create directories
raw_dir = '/kaggle/working/LinguaFranca/data/raw/2wikimultihopqa'
os.makedirs(raw_dir, exist_ok=True)
os.makedirs('/kaggle/working/LinguaFranca/data/processed', exist_ok=True)

# Download generated_cot.jsonl from Phase 3 repo
print("Downloading generated_cot.jsonl...")
filepath = hf_hub_download(repo_id="AnishRacherla/LinguaFranca-Phase3", filename="data/2wikimultihopqa/generated_cot.jsonl", repo_type="dataset")
shutil.copy(filepath, os.path.join(raw_dir, 'generated_cot.jsonl'))
print("Downloaded generated_cot.jsonl!")

### 3. Load Config & Llama 3.2

In [ ]:
import yaml, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

with open('configs/data_config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['data']['raw_dir'] = '/kaggle/working/LinguaFranca/data/raw'
cfg['data']['processed_dir'] = '/kaggle/working/LinguaFranca/data/processed'
cfg['matching']['wikidata_aliases_path'] = '/kaggle/working/LinguaFranca/data/raw/wikidata_aliases.json'

print("Loading Llama model for fallback judge...")
tokenizer = AutoTokenizer.from_pretrained(cfg['model']['name'], token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    cfg['model']['name'],
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN
)

### 4. Run updated label_hops.py

In [ ]:
import json
from pathlib import Path
from utils.matching import EntityMatcher
from utils.wikidata_aliases import load_alias_table
from phase1_dataset.label_hops import label_example, make_llm_judge

cot_path = Path(raw_dir) / 'generated_cot.jsonl'
with open(cot_path) as f:
    records = [json.loads(l) for l in f if l.strip()]
print('Loaded', len(records), 'CoT records')

match_cfg = cfg['matching']
aliases = load_alias_table(match_cfg['wikidata_aliases_path'])
matcher = EntityMatcher(
    aliases=aliases,
    sbert_model_name=match_cfg['sbert_model'],
    sbert_threshold=match_cfg['sbert_threshold'],
    llm_judge=make_llm_judge(model, tokenizer),
    llm_budget=match_cfg['llm_fallback_budget'],
)

labeled = [label_example(ex, matcher) for ex in records]

labeled_path = Path(raw_dir) / 'labeled.jsonl'
with open(labeled_path, 'w', encoding='utf-8') as f:
    for ex in labeled: f.write(json.dumps(ex, ensure_ascii=False) + '\n')
print('Clean Labeled data saved to:', labeled_path)

# Verify 0 blanks
blank = sum(1 for e in labeled for h in e.get("hops", []) if not h.get("bridging_entity_gold", "").strip())
print(f"VERIFICATION: {blank} blank gold entities found in labeled.jsonl!")

### 5. Regenerate Counterfactuals and Splits

In [ ]:
import random
from phase1_dataset.counterfactuals import run_counterfactuals

# 1. Counterfactuals
augmented_path = run_counterfactuals(cfg, hf_token=HF_TOKEN)

# 2. Splits
with open(augmented_path, encoding='utf-8') as f:
    all_ex = [json.loads(l) for l in f]

rng = random.Random(cfg.get('seed', 42))
ids = list({e['id'] for e in all_ex}); rng.shuffle(ids)
n = len(ids); t = int(n*0.6); v = int(n*0.2)
tr_ids = set(ids[:t]); va_ids = set(ids[t:t+v]); te_ids = set(ids[t+v:])

train = [e for e in all_ex if e['id'] in tr_ids]
val   = [e for e in all_ex if e['id'] in va_ids]
test  = [e for e in all_ex if e['id'] in te_ids]

processed = Path(cfg['data']['processed_dir'])
for name, split in [('train', train), ('val', val), ('test', test)]:
    with open(processed / (name + '.jsonl'), 'w', encoding='utf-8') as f:
        for e in split: f.write(json.dumps(e, ensure_ascii=False) + '\n')
print('train:', len(train), '| val:', len(val), '| test:', len(test))

### 6. Upload Clean Files back to Hugging Face

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "AnishRacherla/LinguaFranca-Phase3"

files_to_upload = [
    (str(labeled_path), "data/2wikimultihopqa/labeled.jsonl"),
    (str(augmented_path), "data/2wikimultihopqa/augmented.jsonl"),
    (str(processed / "train.jsonl"), "data/2wikimultihopqa/train.jsonl"),
    (str(processed / "val.jsonl"), "data/2wikimultihopqa/val.jsonl"),
    (str(processed / "test.jsonl"), "data/2wikimultihopqa/test.jsonl"),
]

for local_path, repo_path in files_to_upload:
    print(f"Uploading {repo_path}...")
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=repo_path,
        repo_id=repo_id,
        repo_type="dataset",
        commit_message=f"Update {repo_path} with 0 blank gold entities"
    )

print("All clean files successfully uploaded to Hugging Face!")